In [10]:
import numpy as np

import toponetx as tnx
from toponetx.utils.normalization import compute_bunch_normalized_matrices,compute_laplacian_normalized_matrix
from toponetx.utils.normalization import *
import scipy.sparse.linalg as spl
from numpy.linalg import pinv
from scipy import sparse
from scipy.sparse import csr_matrix

In [11]:
edge_set = [[1, 2], [1, 3], [2,3], [2,4], [2,6], [3,4], [4,5], [4,7], [5,6], [5,7]]
face_set = [[1, 2, 3], [2, 3, 4]]

ex2_sc = tnx.SimplicialComplex(edge_set + face_set)

In [12]:
ex2_sc.simplices

SimplexView([(1,), (2,), (3,), (4,), (6,), (5,), (7,), (1, 2), (1, 3), (2, 3), (2, 4), (2, 6), (3, 4), (4, 5), (4, 7), (5, 6), (5, 7), (1, 2, 3), (2, 3, 4)])

In [13]:
ex2_incidence1 = np.array(ex2_sc.incidence_matrix(rank=1).todense())
ex2_incidence1

array([[-1., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 1.,  0., -1., -1., -1.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  1.,  1.,  0.,  0., -1.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  1.,  0.,  1., -1., -1.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -1., -1.],
       [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  1.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  1.]], dtype=float32)

In [14]:
ex2_sc.coincidence_matrix(1).todense()

matrix([[-1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [-1.,  0.,  1.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  0.,  1.,  0.,  0.,  0.],
        [ 0., -1.,  0.,  0.,  0.,  1.,  0.],
        [ 0.,  0., -1.,  1.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  1.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  0.,  0.,  1.],
        [ 0.,  0.,  0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  0.,  1.]], dtype=float32)

In [15]:
ex2_incidence2 = np.array(ex2_sc.incidence_matrix(rank=2).todense())
ex2_incidence2

array([[ 1.,  0.],
       [-1.,  0.],
       [ 1.,  1.],
       [ 0., -1.],
       [ 0.,  0.],
       [ 0.,  1.],
       [ 0.,  0.],
       [ 0.,  0.],
       [ 0.,  0.],
       [ 0.,  0.]], dtype=float32)

In [16]:
ex2_sc.down_laplacian_matrix(rank=1).todense().A.shape

(10, 10)

In [17]:
B1_norm, B1T_norm, B2_norm, B2T_norm = compute_bunch_normalized_matrices(ex2_incidence1, ex2_incidence2)

In [28]:
B1_norm

array([[-0.25      , -0.25      ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.1       ,  0.        , -0.1       , -0.1       , -0.1       ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.125     ,  0.125     ,  0.        ,  0.        ,
        -0.125     ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.125     ,  0.        ,
         0.125     , -0.125     , -0.125     ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.16666667,  0.        , -0.16666667, -0.16666667],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.25      ,
         0.        ,  0.        ,  0.        ,  0.25      ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.25      

In [26]:
B1T_norm

array([[-0.25      ,  0.1       ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [-0.25      ,  0.        ,  0.125     ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        , -0.2       ,  0.25      ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [ 0.        , -0.1       ,  0.        ,  0.125     ,  0.        ,
         0.        ,  0.        ],
       [ 0.        , -0.1       ,  0.        ,  0.        ,  0.        ,
         0.25      ,  0.        ],
       [ 0.        ,  0.        , -0.125     ,  0.125     ,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.125     ,  0.16666667,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.125     ,  0.        ,
         0.        ,  0.25      ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -0.16666667,
         0.25      ,  0.        ],
       [ 0.        ,  0.    

In [18]:
def normalized_high_order_adjcency(B1, B2):
    # L0
    L0 = B1@B1.T
    B1_sum = np.abs(B1).sum(axis=1)
    D0 = sparse.diags(B1_sum.reshape(-1), 0)
    B1_sum_inv = 1./B1_sum
    B1_sum_inv[np.isinf(B1_sum_inv) | np.isneginf(B1_sum_inv)] = 0
    D0_inv = np.diag(B1_sum_inv.reshape(-1), 0)
    L0 = D0_inv@L0
    L0factor = (-1)*np.diag((1/(B1_sum_inv + 1)).reshape(-1), 0)
    L0bias = np.eye(D0.shape[0])
    L0 = L0factor@L0 + L0bias

    #L1
    A_1u = _compute_D2(B2) - B2@_compute_D3(B2)@B2.T

    A_1d = pinv(_compute_D2(B2)) - ex2_incidence1.T@pinv(_compute_D1(B1, _compute_D2(B2)))@B1
    
    A_1u_norm = (
            A_1u
            +
            np.eye(A_1u.shape[0])
    )@np.diag(1/_compute_D2(B2).diagonal()+1, 0)
    
    A_1d_norm = (
        _compute_D2(B2)
        +
        np.eye(_compute_D2(B2).shape[0])
    )@(A_1d + np.eye(A_1d.shape[0]))
    
    L1 = A_1u_norm + A_1d_norm
    
    #L2
    A_2d = np.eye(B2.shape[1]) - B2.T@pinv(_compute_D5(B2))@B2
    
    L2 = (2*np.eye(B2.shape[1]))@(A_2d + np.eye(A_2d.shape[0]))

    return L0, L1, L2

In [19]:
A_2d = np.eye(ex2_incidence2.shape[1]) + ex2_incidence2.T@pinv(_compute_D5(ex2_incidence2))@ex2_incidence2

In [21]:
L0, L1, L2 = normalized_high_order_adjcency(ex2_incidence1, ex2_incidence2)

In [216]:
A_1u = _compute_D2(ex2_incidence2) - ex2_incidence2@_compute_D3(ex2_incidence2)@ex2_incidence2.T

In [24]:
L2

array([[-1., -1.],
       [-1., -1.]])

In [201]:
pinv(np.eye(ex2_incidence2.shape[1]))

array([[1., 0.],
       [0., 1.]])

In [206]:
A_2d = np.eye(ex2_incidence2.shape[1]) - ex2_incidence2.T@pinv(_compute_D5(ex2_incidence2))@ex2_incidence2

L2 = (2*np.eye(ex2_incidence2.shape[1]))@(A_2d + np.eye(A_2d.shape[0]))

In [207]:
A_2d

array([[-1.5, -0.5],
       [-0.5, -1.5]])

In [202]:
(A_2d + np.eye(A_2d.shape[0]))

array([[-0.5, -0.5],
       [-0.5, -0.5]])

## Degree matrix

In [103]:
D2_pre = np.diag(np.abs(ex2_incidence2).sum(axis = 1).reshape(-1))
identity_matrix_1 = np.eye(D2_pre.shape[0]) # 10x10
D2 = np.maximum(D2_pre, identity_matrix)


D2_tile_pre = np.diag(np.abs(ex2_incidence1).sum(axis = 1).reshape(-1))
identity_matrix_2 = np.eye(D2_tile_pre.shape[0]) # 7x7
D2_tile = np.maximum(D2_tile_pre, identity_matrix_2)

D1 = 2 * np.diag((np.abs(ex2_incidence1) @ D2).sum(axis = 1).reshape(-1))

In [145]:
identity_matrix_3 = np.eye(2)
D3 = round(1/3, 3) * identity_matrix_3
D3_tile = identity_matrix_3
D4 = identity_matrix_3
D5 = np.diag(np.abs(ex2_incidence2).sum(axis = 1).reshape(-1))

In [186]:
identity_matrix_3

array([[1., 0.],
       [0., 1.]])

In [182]:
np.abs(ex2_incidence1) # nodes X edges

array([[1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 1., 1., 1., 0., 0., 0., 0., 0.],
       [0., 1., 1., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 1., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 1., 1.],
       [0., 0., 0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 1.]], dtype=float32)

In [183]:
np.abs(ex2_incidence1) @ D2

array([[1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 2., 1., 1., 0., 0., 0., 0., 0.],
       [0., 1., 2., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 1., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 1., 1.],
       [0., 0., 0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 1.]])

In [189]:
ex2_incidence2

array([[ 1.,  0.],
       [-1.,  0.],
       [ 1.,  1.],
       [ 0., -1.],
       [ 0.,  0.],
       [ 0.,  1.],
       [ 0.,  0.],
       [ 0.,  0.],
       [ 0.,  0.],
       [ 0.,  0.]], dtype=float32)

In [184]:
D2 # edges X edges

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 2., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [161]:
np.linalg.det(D5)

0.0

## Normalized up or down laplacian

In [ ]:
norm_rank0_up_laplacian = ex2_incidence1 @ D3_tile @ ex2_incidence1.T @ np.diag(1/np.diag(D2_tile))
norm_rank1_up_laplacian = D2 @ ex2_incidence1.T @ np.linalg.inv(D1) @ ex2_incidence1

norm_rank1_down_laplacian = ex2_incidence2 @ D3 @ ex2_incidence2.T @ np.linalg.inv(D2)
norm_rank2_down_laplacian = 

In [123]:
# Example matrix
matrix = np.array([[1, 2], [3, 4]])

# Compute the inverse of the matrix
inverse_matrix = np.linalg.inv(np.diag([2, 5, -1]))

In [130]:
ex2_incidence1 @ D3 @ ex2_incidence1.T @ np.linalg.inv(D2)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10 is different from 7)

In [142]:
(ex2_incidence2 @ D3 @ ex2_incidence2.T @ np.linalg.inv(D2)).shape

(10, 10)

In [152]:
ex2_incidence2

array([[ 1.,  0.],
       [-1.,  0.],
       [ 1.,  1.],
       [ 0., -1.],
       [ 0.,  0.],
       [ 0.,  1.],
       [ 0.,  0.],
       [ 0.,  0.],
       [ 0.,  0.],
       [ 0.,  0.]], dtype=float32)

In [153]:
np.diag(1/np.diag(D5),)

/var/folders/2s/s7d6qfbj0fb415tqwkd_3qkm0000gn/T/ipykernel_85378/404936194.py:1: RuntimeWarning: divide by zero encountered in divide
  np.diag(1/np.diag(D5))


array([[1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , inf, 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , inf, 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , inf, 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , inf, 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , inf]], dtype=float32)

In [165]:
(D4 @ ex2_incidence2.T).shape

(2, 10)

In [ ]:
# so D5 is 10 x 10

In [166]:
ex2_incidence2.shape

(10, 2)

In [155]:
np.linalg.inv(D5)

LinAlgError: Singular matrix

In [50]:
import numpy as np

# Example diagonal matrix
diagonal_matrix = np.diag([2, 5, -1])

# Identity matrix of the same size
identity_matrix = np.eye(diagonal_matrix.shape[0])

# Compare the diagonal elements and take the maximum
max_diagonal_elements = np.maximum(np.diag(diagonal_matrix), np.diag(identity_matrix))

# Create a new diagonal matrix with the maximum elements
max_diagonal_matrix = np.diag(max_diagonal_elements)

print("Original Diagonal Matrix:")
print(diagonal_matrix)
print("\nIdentity Matrix:")
print(identity_matrix)
print("\nMax Diagonal Matrix:")
print(max_diagonal_matrix)


Original Diagonal Matrix:
[[ 2  0  0]
 [ 0  5  0]
 [ 0  0 -1]]

Identity Matrix:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Max Diagonal Matrix:
[[2. 0. 0.]
 [0. 5. 0.]
 [0. 0. 1.]]


In [48]:
np.diag(np.abs(ex2_incidence2).sum(axis = 1).reshape(-1))

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 2., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [19]:
# up laplacian rank 1: edge to incident faces to incident edges
up_laplacian_1 = ex2_sc.up_laplacian_matrix(rank=1).todense()
print(up_laplacian_1)

[[ 1. -1.  1.  0.  0.  0.  0.  0.  0.  0.]
 [-1.  1. -1.  0.  0.  0.  0.  0.  0.  0.]
 [ 1. -1.  2. -1.  0.  1.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0. -1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1. -1.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]]


In [20]:
# down laplacian rank 1: edges to incident nodes to incident edges
down_laplacian_1 = ex2_sc.down_laplacian_matrix(rank=1).todense()

print(down_laplacian_1)

[[ 2.  1. -1. -1. -1.  0.  0.  0.  0.  0.]
 [ 1.  2.  1.  0.  0. -1.  0.  0.  0.  0.]
 [-1.  1.  2.  1.  1. -1.  0.  0.  0.  0.]
 [-1.  0.  1.  2.  1.  1. -1. -1.  0.  0.]
 [-1.  0.  1.  1.  2.  0.  0.  0.  1.  0.]
 [ 0. -1. -1.  1.  0.  2. -1. -1.  0.  0.]
 [ 0.  0.  0. -1.  0. -1.  2.  1. -1. -1.]
 [ 0.  0.  0. -1.  0. -1.  1.  2.  0.  1.]
 [ 0.  0.  0.  0.  1.  0. -1.  0.  2.  1.]
 [ 0.  0.  0.  0.  0.  0. -1.  1.  1.  2.]]


In [21]:
# down laplacian rank 2: face to incident edges to incident faces
down_laplacian_2 = ex2_sc.down_laplacian_matrix(rank=2).todense()

print(down_laplacian_2)

[[3. 1.]
 [1. 3.]]
